# Comparación de la semana · obstáculos que cambian de lugar

Los seis experimentos de esta semana, todos de 500.000 pasos con semilla 0:

| Vista | Sin sampler | Con sampler |
|---|---|---|
| 5×5 | Dos figuras verdes | Dos figuras verdes · Tres figuras de colores |
| 3×3 | Dos figuras verdes | Dos figuras verdes · Tres figuras de colores |

- **Dos figuras verdes** (semana del 24): un cuadrado y un triángulo verdes, uno en cada habitación lateral.
- **Tres figuras de colores** (semana del 25): un cuadrado verde y un triángulo morado en las habitaciones laterales y un rombo amarillo en la del botón.
- En ambos casos las figuras cambian de casilla al comenzar cada partida y se quedan quietas durante ella.
- **Sampler** (`buffer.reward_slices=1`): en cada batch, una de las 16 secuencias se toma alrededor de un momento en que el agente comió una galleta.

Como referencia aparecen en gris los runs **sin obstáculos** de la semana del 15 con la misma configuración. En 3×3 no hay un run sin obstáculos con sampler.

**Usa CPU y ejecuta todo** (Entorno de ejecución → Ejecutar todo). Los tres videos que faltan se crean en la sección 6 y quedan guardados en Drive; la primera vez tarda unos minutos.

## 1. Montar Drive

Usa la cuenta donde están los entrenamientos.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Los runs

Lee las métricas de cada run. Si un entrenamiento se retomó desde un checkpoint, se descarta lo que quedó repetido.

In [ ]:
import json
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta")

OBSTACULOS = {
    "ninguno": "Sin obstáculos (semana del 15)",
    "verdes": "Dos figuras verdes (semana del 24)",
    "colores": "Tres figuras de colores (semana del 25)",
}
COLORES = {"ninguno": "#8a8a8a", "verdes": "#257de0", "colores": "#f47230"}

# (vista, sampler, obstáculos, carpeta en Drive)
RUNS = [
    ("5×5", False, "ninguno", "semana15/cookie_partial_hall5_view5_s0_base"),
    ("5×5", False, "verdes", "semana24/obstaculos_aleatorios5x5/distinct_s0"),
    ("5×5", True, "ninguno", "semana15/cookie_partial_hall5_view5_s0_base_sampler1"),
    ("5×5", True, "verdes", "semana24/obstaculos_aleatorios5x5/distinct_sampler1_s0"),
    ("5×5", True, "colores", "semana25/tres_obstaculos_colores5x5/distinct_sampler1_s0"),
    ("3×3", False, "ninguno", "semana15/cookie_partial_hall5_view3_s0_base"),
    ("3×3", False, "verdes", "semana24/obstaculos_aleatorios3x3/distinct_s0"),
    ("3×3", True, "verdes", "semana24/obstaculos_aleatorios3x3/distinct_sampler1_s0"),
    ("3×3", True, "colores", "semana25/tres_obstaculos_colores3x3/distinct_sampler1_s0"),
]


def leer(carpeta):
    """Evaluaciones y galletas por partida de entrenamiento, como listas de (paso, valor)."""
    evaluaciones, partidas = [], []
    with open(RAIZ / carpeta / "metrics.jsonl") as archivo:
        for linea in archivo:
            fila = json.loads(linea)
            for clave, serie in (("episode/eval_score", evaluaciones), ("episode/score", partidas)):
                if clave in fila:
                    # Al retomar se vuelve a un paso anterior: se borra lo que se repite.
                    while serie and serie[-1][0] >= fila["step"]:
                        serie.pop()
                    serie.append((fila["step"], fila[clave]))
    return evaluaciones, partidas


DATOS = {}
for vista, sampler, obstaculos, carpeta in RUNS:
    nombre = f"{vista} · {'con' if sampler else 'sin'} sampler · {OBSTACULOS[obstaculos]}"
    try:
        DATOS[carpeta] = leer(carpeta)
        evaluaciones, partidas = DATOS[carpeta]
        pasos = max(evaluaciones[-1][0], partidas[-1][0])
        print(f"{nombre}: hasta {pasos:,} pasos".replace(",", "."))
    except FileNotFoundError:
        print(f"{nombre}: NO ENCONTRADO en {carpeta}")

## 3. Galletas que comió el actor mientras entrenaba

Total acumulado de galletas en las partidas con que se entrenó. Es lo que el modelo pudo aprender: sin galletas en los datos, no hay nada que repetir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def graficar(titulo, serie, eje_y, *, marcadores=False):
    figura, ejes = plt.subplots(2, 2, figsize=(11, 7.2), sharex=True, sharey=True, layout="constrained")
    for fila, sampler in enumerate((False, True)):
        for columna, vista in enumerate(("5×5", "3×3")):
            eje = ejes[fila, columna]
            hay_referencia = False
            for v, s, obstaculos, carpeta in RUNS:
                if (v, s) != (vista, sampler) or carpeta not in DATOS:
                    continue
                pasos, valores = serie(*DATOS[carpeta])
                referencia = obstaculos == "ninguno"
                hay_referencia |= referencia
                eje.plot(pasos / 1000, valores, color=COLORES[obstaculos], linewidth=2,
                         linestyle="--" if referencia else "-",
                         marker="o" if marcadores else None, markersize=4, zorder=3 if referencia else 2,
                         label=OBSTACULOS[obstaculos])
            eje.set_title(f"Vista {vista} · {'con' if sampler else 'sin'} sampler", loc="left", fontsize=12)
            if not hay_referencia:
                eje.text(0.02, 0.96, "No hay run sin obstáculos\ncon sampler en 3×3", transform=eje.transAxes,
                         va="top", fontsize=9, color="#666666")
            eje.grid(axis="y", color="#e6e6e6")
            eje.set_axisbelow(True)
            eje.spines[["top", "right"]].set_visible(False)
            eje.spines[["left", "bottom"]].set_color("#999999")
            eje.tick_params(colors="#555555")
            eje.set_xlim(0, 500)
            if fila == 1:
                eje.set_xlabel("Pasos de entrenamiento (miles)")
            if columna == 0:
                eje.set_ylabel(eje_y)
    etiquetas = dict(zip(*reversed(ejes[1, 0].get_legend_handles_labels())))
    figura.legend(etiquetas.values(), etiquetas.keys(), loc="outside lower center", ncol=3, frameon=False, fontsize=11)
    figura.suptitle(titulo, fontsize=14, fontweight="bold")
    plt.show()


def acumuladas(evaluaciones, partidas):
    pasos, galletas = np.array(partidas).T
    return pasos, np.cumsum(galletas)


graficar("Galletas que comió el actor mientras entrenaba", acumuladas, "Galletas acumuladas")

## 4. Evaluación del actor

Cada 25.000 pasos el actor juega cinco partidas eligiendo siempre su acción más probable. Se muestra el promedio de galletas por partida.

In [ ]:
def evaluacion(evaluaciones, partidas):
    return np.array(evaluaciones).T


graficar("Evaluación: galletas por partida", evaluacion, "Galletas por partida (promedio de 5)", marcadores=True)

## 5. Resumen

*Galletas al entrenar* es el total del gráfico de la sección 3. *Primera evaluación con galletas* no cuenta la del paso 0, que es antes de entrenar.

In [ ]:
import pandas as pd

filas = []
for vista, sampler, obstaculos, carpeta in RUNS:
    if carpeta not in DATOS:
        continue
    evaluaciones, partidas = DATOS[carpeta]
    # La evaluación del paso 0 es antes de entrenar: no cuenta.
    primera = next((paso for paso, valor in evaluaciones if paso > 0 and valor > 0), None)
    filas.append({
        "Vista": vista,
        "Sampler": "sí" if sampler else "no",
        "Obstáculos": OBSTACULOS[obstaculos],
        "Pasos": f"{round(max(evaluaciones[-1][0], partidas[-1][0]) / 1000)}k",
        "Galletas al entrenar": int(sum(valor for _, valor in partidas)),
        "Primera evaluación con galletas": f"{primera // 1000}k" if primera is not None else "nunca",
        "Mejor evaluación": max(valor for _, valor in evaluaciones),
        "Evaluación final": evaluaciones[-1][1],
    })

pd.DataFrame(filas).style.hide(axis="index").format(precision=1, decimal=",", thousands=".")

## 6. Crear los videos que faltan (CPU)

Crea el video de la partida de semilla 0 para cada run de esta semana que no lo tenga o cuyo video sea anterior al último checkpoint. Juega el **actor aprendido** del último checkpoint, igual que en las evaluaciones.

Descarga una revisión fija del repositorio, instala las dependencias con PyTorch para CPU y usa el entorno de obstáculos de la semana de cada run. El video queda en la carpeta del run como `agente_episodio0.mp4`, el mismo nombre que usa la sección 7 de los notebooks de entrenamiento. Tarda unos minutos la primera vez y menos de un minuto por video.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

TRABAJO = Path("/content")
REPO = TRABAJO / "r2dreamer-videos"
URL = "https://github.com/FabriRandon/r2dreamer-cookie.git"
REVISION = "bdffa746f32dac6e6d9c7f37fc8f6138a671e117"
SEMILLA_VIDEO = 0
# Cada semana entrenó con su propia versión del entorno de obstáculos.
ENTORNO = {
    "semana24": "notebooks/experimentos-semana-del-24/cookie.py",
    "semana25": "notebooks/experimentos-semana-del-25/cookie.py",
}
VIDEOS = [run for run in RUNS if run[2] != "ninguno"]


def video_de(carpeta):
    archivo = RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4"
    try:
        return archivo if archivo.is_file() and archivo.stat().st_size > 0 else None
    except OSError:
        return None  # Drive puede estar sincronizando el archivo


def al_dia(carpeta):
    """El video está al día si se creó después del último checkpoint."""
    archivo, checkpoint = video_de(carpeta), RAIZ / carpeta / "latest.pt"
    try:
        return archivo is not None and archivo.stat().st_mtime >= checkpoint.stat().st_mtime
    except OSError:
        return False


def nombre(run):
    vista, sampler, obstaculos, _ = run
    return f"{vista} · {'con' if sampler else 'sin'} sampler · {OBSTACULOS[obstaculos]}"


pendientes = []
for run in VIDEOS:
    carpeta = run[3]
    if not (RAIZ / carpeta / "latest.pt").exists():
        print(f"{nombre(run)}: no hay checkpoint en Drive.")
    elif not al_dia(carpeta):
        pendientes.append(run)

if not pendientes:
    print("Los seis videos ya están hechos con el último checkpoint.")
else:
    print(f"Faltan {len(pendientes)} videos. Preparando el código…")
    if not (REPO / ".git").is_dir():
        REPO.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet"], cwd=REPO, check=True)
        subprocess.run(["git", "fetch", "--quiet", "--depth=1", URL, REVISION], cwd=REPO, check=True)
        subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
    PYTHON = REPO / ".venv/bin/python"
    if not PYTHON.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        UV = [sys.executable, "-m", "uv"]
        subprocess.run([*UV, "venv", "--quiet", "--python", "3.11", ".venv"], cwd=REPO, check=True)
        (REPO / "cpu.txt").write_text("torch==2.8.0+cpu\n")
        subprocess.run([*UV, "pip", "install", "--quiet", "--python", str(PYTHON),
                        "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                        "--index-strategy", "unsafe-best-match", "--constraint", "cpu.txt",
                        "-e", ".[cookie]"], cwd=REPO, check=True)
    temporal = Path(tempfile.mkdtemp(prefix="videos-", dir=TRABAJO))
    for run in pendientes:
        carpeta = run[3]
        print(f"Creando video: {nombre(run)}")
        shutil.copyfile(REPO / ENTORNO[carpeta.split("/")[0]], REPO / "envs/cookie.py")
        provisional = temporal / f"{carpeta.replace('/', '_')}.mp4"
        proceso = subprocess.run([str(PYTHON), "watch.py", "--logdir", str(RAIZ / carpeta),
                                  "--out", str(provisional), "--seed", str(SEMILLA_VIDEO)],
                                 cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                                 text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if proceso.returncode == 0 and provisional.exists():
            shutil.copyfile(provisional, RAIZ / carpeta / f"agente_episodio{SEMILLA_VIDEO}.mp4")
            print("  ", proceso.stdout.strip().splitlines()[-1].split(" Video en ")[0])
        else:
            print("   No se pudo crear. Vuelve a ejecutar esta celda.")
            print("\n".join(proceso.stdout.splitlines()[-10:]))

## 7. Ver los videos

A la izquierda, el mapa completo; a la derecha, lo que recibe el agente. El encabezado indica las galletas y los pasos entrenados del checkpoint.

En el 3×3 de dos figuras verdes con sampler, el último checkpoint guardado en Drive es de 370.000 pasos, aunque sus métricas llegan a 500.000. Su evaluación fue 0 durante todo el run.

In [ ]:
import html
from IPython.display import HTML, Video, display


def tarjeta(run):
    vista, sampler, obstaculos, carpeta = run
    titulo = f"<h4>{html.escape(OBSTACULOS[obstaculos])}</h4><p>{'Con' if sampler else 'Sin'} sampler</p>"
    archivo = video_de(carpeta)
    if archivo is None:
        return f'<div class="tarjeta">{titulo}<p>No hay video. Ejecuta la sección 6.</p></div>'
    try:
        reproductor = Video(str(archivo), embed=True)._repr_html_()
    except OSError:
        return f'<div class="tarjeta">{titulo}<p>Drive aún no pudo leer el video. Vuelve a ejecutar esta celda.</p></div>'
    final = DATOS[carpeta][0][-1][1] if carpeta in DATOS else None
    nota = f"Evaluación final: {final:.1f} galletas por partida.".replace(".", ",", 1) if final is not None else ""
    if not al_dia(carpeta):
        nota += " ⚠️ Video anterior al último checkpoint: ejecuta la sección 6."
    return f'<div class="tarjeta">{titulo}{reproductor}<p><small>{html.escape(nota)}</small></p></div>'


estilos = (
    "<style>"
    ".vista{max-width:1200px;margin:0 0 28px}"
    ".vista h3{margin:0 0 10px}"
    ".fila{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:16px}"
    ".tarjeta{border:1px solid #8885;border-radius:8px;padding:12px}"
    ".tarjeta h4{margin:0}"
    ".tarjeta p{margin:2px 0 8px}"
    ".tarjeta video{width:100%;height:auto}"
    "</style>"
)
bloques = []
for vista in ("5×5", "3×3"):
    tarjetas = "".join(tarjeta(run) for run in VIDEOS if run[0] == vista)
    bloques.append(f'<div class="vista"><h3>Vista {vista}</h3><div class="fila">{tarjetas}</div></div>')
display(HTML(estilos + "".join(bloques)))